In [1]:
import glob
import pandas as pd 

In [2]:
files = sorted(glob.glob('Datas/*/*.csv')) #every year, every month total should be 12+12+10= 34 
print(len(files))

34


In [3]:
parts = []
for file in files:
    parts.append(pd.read_csv(file, usecols=["started_at"])) # only the start time is needed for daily counts
df = pd.concat(parts, ignore_index=True)

df["started_at"] = pd.to_datetime(df["started_at"], format="ISO8601") # timestamps with and without fractional seconds
df["started_at"] = df["started_at"].dt.tz_convert("Europe/Oslo") # UTC -> Oslo time, so every trip lands on its local day
df["date"] = df["started_at"].dt.date

daily = df.groupby("date").size().reset_index(name="ride_count")
print(daily.shape)
print(daily["ride_count"].sum())

(1034, 2)
3337798


In [4]:
import numpy as np
daily["date"] = pd.to_datetime(daily["date"])
daily["day_type"] = np.where(daily["date"].dt.dayofweek < 5, "Weekday", "Weekend")

season_map = { 12: "Winter", 1: "Winter", 2: "Winter",
               3: "Spring", 4: "Spring", 5: "Spring",   
               6: "Summer", 7: "Summer", 8: "Summer",
               9: "Autumn", 10: "Autumn", 11: "Autumn"}
daily["season"] = daily["date"].dt.month.map(season_map)
daily["year"] = daily["date"].dt.year
print(daily.dtypes)
print(daily.groupby("year").size())
daily.head()

date          datetime64[ns]
ride_count             int64
day_type              object
season                object
year                   int32
dtype: object
year
2024    366
2025    365
2026    303
dtype: int64


,date,ride_count,day_type,season,year
0,2024-01-01,34,Weekday,Winter,2024
1,2024-01-02,136,Weekday,Winter,2024
2,2024-01-03,262,Weekday,Winter,2024
3,2024-01-04,294,Weekday,Winter,2024
4,2024-01-05,269,Weekday,Winter,2024


In [5]:
old = pd.read_csv("daily_2025.csv", parse_dates=["date"])
new = daily[daily["year"] == 2025]
print(len(old), len(new))

check = old.merge(new, on="date", suffixes=("_old", "_new"))
check["diff"] = check["ride_count_new"] - check["ride_count_old"]

print(check[check["diff"] != 0][["date", "ride_count_old", "ride_count_new", "diff"]])
print((check["day_type_old"] == check["day_type_new"]).all())
print((check["season_old"] == check["season_new"]).all())



365 365
        date  ride_count_old  ride_count_new  diff
0 2025-01-01              83              85     2
True
True


In [6]:
may17 = daily[(daily["date"].dt.month == 5) & (daily["date"].dt.day == 17)]
print(may17)

daily["closed"] = daily["date"] == "2025-05-17"
print(daily["closed"].sum())

          date  ride_count day_type  season  year
137 2024-05-17        1585  Weekday  Spring  2024
502 2025-05-17           4  Weekend  Spring  2025
1


In [7]:
import holidays

no_holidays = holidays.Norway(years=[2024, 2025, 2026])
for day, name in sorted(no_holidays.items()):
    print(day, name)

daily["is_holiday"] = daily["date"].isin(pd.to_datetime(list(no_holidays.keys())))
print(daily["is_holiday"].sum())
print(daily.groupby("year")["is_holiday"].sum())

print(daily[~daily["closed"]].groupby(["day_type", "is_holiday"])["ride_count"].agg(["mean", "count"]).round(0))

2024-01-01 Første nyttårsdag
2024-03-28 Skjærtorsdag
2024-03-29 Langfredag
2024-03-31 Første påskedag
2024-04-01 Andre påskedag
2024-05-01 Arbeidernes dag
2024-05-09 Kristi himmelfartsdag
2024-05-17 Grunnlovsdag
2024-05-19 Første pinsedag
2024-05-20 Andre pinsedag
2024-12-25 Første juledag
2024-12-26 Andre juledag
2025-01-01 Første nyttårsdag
2025-04-17 Skjærtorsdag
2025-04-18 Langfredag
2025-04-20 Første påskedag
2025-04-21 Andre påskedag
2025-05-01 Arbeidernes dag
2025-05-17 Grunnlovsdag
2025-05-29 Kristi himmelfartsdag
2025-06-08 Første pinsedag
2025-06-09 Andre pinsedag
2025-12-25 Første juledag
2025-12-26 Andre juledag
2026-01-01 Første nyttårsdag
2026-04-02 Skjærtorsdag
2026-04-03 Langfredag
2026-04-05 Første påskedag
2026-04-06 Andre påskedag
2026-05-01 Arbeidernes dag
2026-05-14 Kristi himmelfartsdag
2026-05-17 Grunnlovsdag
2026-05-24 Første pinsedag
2026-05-25 Andre pinsedag
2026-12-25 Første juledag
2026-12-26 Andre juledag
33
year
2024    12
2025    12
2026     9
Name: is_ho

In [8]:
daily.to_csv("daily_allyears.csv", index=False)
check_all = pd.read_csv("daily_allyears.csv", parse_dates=["date"])
print(check_all.shape)
print(check_all.dtypes)
print(check_all["ride_count"].sum())

(1034, 7)
date          datetime64[ns]
ride_count             int64
day_type              object
season                object
year                   int64
closed                  bool
is_holiday              bool
dtype: object
3337798
